## Ultra-Light <1B 

### 1. iFaz/llama32_1B_en_emo_v1

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("iFaz/llama32_3B_en_emo_v1")
model = AutoModelForCausalLM.from_pretrained("iFaz/llama32_3B_en_emo_v1")




input_text = "How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5211.84it/s]


KeyboardInterrupt: 

In [ ]:
# Cell 2: The Matrix Testing Script

models_to_test = [
    "Qwen/Qwen2.5-0.5B",
    "meta-llama/Llama-3.2-1B",
    # ... add your chosen 15 models here
]

test_prompts = [
    "How many calories does a average 35-year-old woman who weighs 70kg and is 180 cm tall need per day?"
]

results = {}

for model_id in models_to_test:
    print(f"\n--- Testing Model: {model_id} ---")
    try:
        # 1. Load Tokenizer and Model
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)
        
        results[model_id] = []
        
        # 2. Run Inputs
        for prompt in test_prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            
            start_time = time.time()
            outputs = model.generate(**inputs, max_new_tokens=50)
            end_time = time.time()
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            results[model_id].append({
                "prompt": prompt,
                "response": response,
                "time_taken": round(end_time - start_time, 2)
            })
            print(f"Prompt: {prompt}\nResponse: {response}\nTime: {round(end_time - start_time, 2)}s\n")
            
    except Exception as e:
        print(f"Failed to load or run {model_id}: {e}")
        
    finally:
        # 3. CRITICAL: Clear memory before the next model
        if 'model' in locals():
            del model
        if 'tokenizer' in locals():
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()